# EDA complet du dataset tickets (orientation RAG)

Ce notebook realise une analyse exploratoire complete du fichier CSV:
- qualite et structure des donnees
- distributions metier (type, queue, priorite, langue, version, tags)
- analyse texte (longueurs, vocabulaire, duplications)
- verification de la preparation pour un systeme RAG

Objectif final: identifier les risques de qualite et les actions de pre-processing avant indexation/retrieval.


In [ ]:
# Parametres
from pathlib import Path

CSV_PATH = Path('../data/aa_dataset-tickets-multi-lang-5-2-50-version.csv')
OUTPUT_DIR = Path('../reports/eda_rag')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TOP_K = 20
CHUNK_SIZE_CHARS = 800
CHUNK_OVERLAP_CHARS = 120

print('CSV_PATH:', CSV_PATH.resolve())
print('OUTPUT_DIR:', OUTPUT_DIR.resolve())


In [ ]:
# Imports
import re
import math
import json
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)


In [ ]:
# Chargement
if not CSV_PATH.exists():
    raise FileNotFoundError(f'Fichier introuvable: {CSV_PATH}')

df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
print('Colonnes:', list(df.columns))
df.head(3)


In [ ]:
# Apercu global
print('Info:')
df.info()

print('
Apercu statistique (numerique + texte):')
display(df.describe(include='all').T.head(20))


In [ ]:
# Nettoyage minimal pour EDA (sans modifier le fichier source)
raw_df = df.copy()

text_cols_candidates = ['subject', 'body', 'answer']
text_cols = [c for c in text_cols_candidates if c in df.columns]

for c in text_cols:
    df[c] = df[c].astype(str).str.replace('\r', ' ', regex=False).str.replace('\n', ' ', regex=False).str.strip()

cat_cols = [c for c in ['type', 'queue', 'priority', 'language', 'version'] if c in df.columns]
for c in cat_cols:
    df[c] = df[c].astype(str).str.strip()

# Detection colonnes tag
tag_cols = [c for c in df.columns if c.lower().startswith('tag_')]
print('Colonnes texte:', text_cols)
print('Colonnes categories:', cat_cols)
print('Colonnes tags:', tag_cols)


## 1) Qualite des donnees

In [ ]:
# Valeurs manquantes / vides
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values('missing_pct', ascending=False)

empty = pd.Series({
    col: ((df[col].astype(str).str.strip() == '') if col in df.columns else pd.Series(dtype=bool)).sum()
    for col in df.columns
}, name='empty_count')

quality = missing.join(empty)
quality['empty_pct'] = (quality['empty_count'] / len(df) * 100).round(2)
display(quality)

quality.to_csv(OUTPUT_DIR / 'missing_empty_summary.csv')


In [ ]:
# Visualisation des taux manquants
plot_df = quality.reset_index().rename(columns={'index': 'column'})
plot_df = plot_df[plot_df['missing_pct'] > 0].sort_values('missing_pct', ascending=False)

plt.figure(figsize=(10, 4))
if len(plot_df) == 0:
    plt.text(0.5, 0.5, 'Aucune valeur manquante detectee', ha='center', va='center')
    plt.axis('off')
else:
    sns.barplot(data=plot_df, x='column', y='missing_pct', palette='viridis')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Missing (%)')
    plt.xlabel('Colonne')
plt.title('Taux de valeurs manquantes par colonne')
plt.tight_layout()
plt.show()


In [ ]:
# Duplications exactes
subset_cols = [c for c in ['subject', 'body', 'answer'] if c in df.columns]
exact_dup_mask = df.duplicated(subset=subset_cols, keep=False) if subset_cols else pd.Series([False]*len(df))
exact_dup_count = int(exact_dup_mask.sum())

print('Colonnes utilisees pour duplication exacte:', subset_cols)
print('Lignes dupliquees (exact match):', exact_dup_count)
print('Pourcentage duplique:', round(exact_dup_count / len(df) * 100, 2), '%')

if exact_dup_count > 0:
    display(df.loc[exact_dup_mask, subset_cols + [c for c in ['language', 'type', 'queue'] if c in df.columns]].head(10))


In [ ]:
# Duplications proches (empreinte texte normalisee)
def normalize_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

if {'subject', 'body'}.issubset(df.columns):
    merged_for_hash = (df['subject'].fillna('') + ' ' + df['body'].fillna('')).astype(str)
elif 'body' in df.columns:
    merged_for_hash = df['body'].fillna('').astype(str)
else:
    merged_for_hash = pd.Series([''] * len(df))

norm_hash = merged_for_hash.map(normalize_text)
near_dup_mask = norm_hash.duplicated(keep=False)
near_dup_count = int(near_dup_mask.sum())

print('Lignes potentiellement dupliquees (normalisation simple):', near_dup_count)
print('Pourcentage potentiel:', round(near_dup_count / len(df) * 100, 2), '%')


## 2) Distributions metier et metadata

In [ ]:
# Distribution des variables categorielles principales
for col in ['type', 'queue', 'priority', 'language', 'version']:
    if col not in df.columns:
        continue
    vc = df[col].value_counts(dropna=False)
    out = pd.DataFrame({
        col: vc.index,
        'count': vc.values,
        'pct': (vc.values / len(df) * 100).round(2)
    })
    print(f'\n=== {col} ===')
    display(out.head(15))
    out.to_csv(OUTPUT_DIR / f'distribution_{col}.csv', index=False)


In [ ]:
# Plots des categories
main_cats = [c for c in ['type', 'queue', 'priority', 'language'] if c in df.columns]
if main_cats:
    fig, axes = plt.subplots(len(main_cats), 1, figsize=(11, 3.5 * len(main_cats)))
    if len(main_cats) == 1:
        axes = [axes]
    for ax, col in zip(axes, main_cats):
        tmp = df[col].value_counts().head(TOP_K)
        sns.barplot(x=tmp.values, y=tmp.index, ax=ax, palette='mako')
        ax.set_title(f'Distribution de {col}')
        ax.set_xlabel('Count')
        ax.set_ylabel(col)
    plt.tight_layout()
    plt.show()


In [ ]:
# Analyse des tags
if tag_cols:
    tags_long = (
        df[tag_cols]
        .replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
        .melt(value_name='tag')
        .dropna(subset=['tag'])
    )
    tags_long['tag'] = tags_long['tag'].astype(str).str.strip()
    tag_dist = tags_long['tag'].value_counts().reset_index()
    tag_dist.columns = ['tag', 'count']
    tag_dist['pct_rows'] = (tag_dist['count'] / len(df) * 100).round(2)

    display(tag_dist.head(30))
    tag_dist.to_csv(OUTPUT_DIR / 'tags_distribution.csv', index=False)

    plt.figure(figsize=(10, 6))
    top_tags = tag_dist.head(TOP_K)
    sns.barplot(data=top_tags, x='count', y='tag', palette='crest')
    plt.title(f'Top {TOP_K} tags')
    plt.xlabel('Count')
    plt.ylabel('Tag')
    plt.tight_layout()
    plt.show()
else:
    print('Aucune colonne tag_* detectee.')


## 3) Analyse texte

In [ ]:
# Metriques de longueur texte
for col in text_cols:
    df[f'{col}_chars'] = df[col].fillna('').astype(str).str.len()
    df[f'{col}_words'] = df[col].fillna('').astype(str).str.split().map(len)

length_cols = [c for c in df.columns if c.endswith('_chars') or c.endswith('_words')]
display(df[length_cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)


In [ ]:
# Distribution des longueurs du body et de la reponse
plot_targets = [c for c in ['body_words', 'answer_words'] if c in df.columns]
if plot_targets:
    fig, axes = plt.subplots(1, len(plot_targets), figsize=(7 * len(plot_targets), 4))
    if len(plot_targets) == 1:
        axes = [axes]
    for ax, col in zip(axes, plot_targets):
        sns.histplot(df[col], bins=50, kde=True, ax=ax, color='#2a9d8f')
        ax.set_title(f'Distribution {col}')
        ax.set_xlim(left=0)
    plt.tight_layout()
    plt.show()


In [ ]:
# Longueur moyenne par langue
if 'language' in df.columns and 'body_words' in df.columns:
    by_lang = (
        df.groupby('language', dropna=False)['body_words']
        .agg(['count', 'mean', 'median', 'std'])
        .sort_values('count', ascending=False)
    )
    display(by_lang)
    by_lang.to_csv(OUTPUT_DIR / 'body_length_by_language.csv')

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x='language', y='body_words', showfliers=False)
    plt.title('Distribution body_words par langue (sans outliers visuels)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
# Top mots (simple, sans NLP lourd)
STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'for', 'on', 'with', 'is', 'are', 'i', 'we',
    'le', 'la', 'les', 'de', 'des', 'du', 'et', 'ou', 'dans', 'pour', 'avec', 'est', 'je', 'nous',
    'der', 'die', 'das', 'und', 'oder', 'mit', 'ist', 'ich', 'wir'
}

def top_terms(series, top_n=30):
    tokens = []
    for txt in series.fillna('').astype(str):
        words = re.findall(r'\b\w+\b', txt.lower())
        words = [w for w in words if len(w) > 2 and not w.isdigit() and w not in STOPWORDS]
        tokens.extend(words)
    return Counter(tokens).most_common(top_n)

if 'body' in df.columns:
    top_global = pd.DataFrame(top_terms(df['body'], top_n=50), columns=['token', 'count'])
    display(top_global.head(30))
    top_global.to_csv(OUTPUT_DIR / 'top_terms_body.csv', index=False)


## 4) EDA cible RAG (readiness checks)

In [ ]:
# Construction d'un document source pour retrieval
df_rag = df.copy()

def safe_col(col):
    return df_rag[col].fillna('').astype(str) if col in df_rag.columns else pd.Series(['']*len(df_rag))

df_rag['doc_text'] = (
    'Subject: ' + safe_col('subject') + '\n' +
    'Body: ' + safe_col('body') + '\n' +
    'Type: ' + safe_col('type') + ' | Queue: ' + safe_col('queue') + ' | Priority: ' + safe_col('priority') +
    ' | Language: ' + safe_col('language')
)

# Approximation token count (simple): mots * 1.3
word_counts = df_rag['doc_text'].str.split().map(len)
df_rag['doc_tokens_est'] = (word_counts * 1.3).round().astype(int)

display(df_rag[['doc_text', 'doc_tokens_est']].head(3))
print('Token est. moyenne:', round(df_rag['doc_tokens_est'].mean(), 2))
print('Token est. p95:', int(df_rag['doc_tokens_est'].quantile(0.95)))


In [ ]:
# Simulation de chunking (caracteres)
def estimate_chunks(n_chars, chunk_size=CHUNK_SIZE_CHARS, overlap=CHUNK_OVERLAP_CHARS):
    if n_chars <= 0:
        return 0
    if n_chars <= chunk_size:
        return 1
    step = max(1, chunk_size - overlap)
    return 1 + math.ceil((n_chars - chunk_size) / step)

if 'doc_text' in df_rag.columns:
    df_rag['doc_chars'] = df_rag['doc_text'].str.len()
    df_rag['n_chunks_est'] = df_rag['doc_chars'].map(estimate_chunks)

    display(df_rag['n_chunks_est'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

    plt.figure(figsize=(9, 4))
    sns.histplot(df_rag['n_chunks_est'], bins=30, kde=False, color='#264653')
    plt.title('Distribution du nombre de chunks estimes par document')
    plt.xlabel('n_chunks_est')
    plt.tight_layout()
    plt.show()


In [ ]:
# Couverture metadata utile pour filtres RAG
meta_cols = [c for c in ['type', 'queue', 'priority', 'language', 'version'] if c in df_rag.columns]
meta_coverage = pd.DataFrame({
    'field': meta_cols,
    'non_null_pct': [round((df_rag[c].notna().mean() * 100), 2) for c in meta_cols],
    'non_empty_pct': [round(((df_rag[c].astype(str).str.strip() != '').mean() * 100), 2) for c in meta_cols],
    'n_unique': [df_rag[c].nunique(dropna=True) for c in meta_cols]
})

display(meta_coverage)
meta_coverage.to_csv(OUTPUT_DIR / 'metadata_coverage.csv', index=False)


In [ ]:
# Cohesion question/reponse (proxy simple de qualite)
# Mesure: overlap Jaccard entre mots du body et de la reponse

def token_set(txt):
    words = re.findall(r'\b\w+\b', str(txt).lower())
    words = [w for w in words if len(w) > 2 and w not in STOPWORDS and not w.isdigit()]
    return set(words)

if {'body', 'answer'}.issubset(df_rag.columns):
    jaccards = []
    for b, a in zip(df_rag['body'], df_rag['answer']):
        sb, sa = token_set(b), token_set(a)
        if not sb and not sa:
            j = np.nan
        else:
            union = len(sb | sa)
            inter = len(sb & sa)
            j = inter / union if union else np.nan
        jaccards.append(j)

    df_rag['body_answer_jaccard'] = jaccards

    display(df_rag['body_answer_jaccard'].describe(percentiles=[0.1, 0.5, 0.9, 0.95]))

    plt.figure(figsize=(9, 4))
    sns.histplot(df_rag['body_answer_jaccard'].dropna(), bins=40, kde=True, color='#e76f51')
    plt.title('Cohesion body/answer (Jaccard token overlap)')
    plt.xlabel('Jaccard overlap')
    plt.tight_layout()
    plt.show()


In [ ]:
# Risque de leakage simple entre train/test (split aleatoire + duplication exacte)
try:
    from sklearn.model_selection import train_test_split
    use_sklearn = True
except Exception:
    use_sklearn = False

if {'subject', 'body'}.issubset(df_rag.columns):
    key = (df_rag['subject'].fillna('') + '||' + df_rag['body'].fillna('')).astype(str)
else:
    key = df_rag['doc_text'].astype(str)

idx = np.arange(len(df_rag))
if use_sklearn:
    train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=RANDOM_STATE)
else:
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(idx)
    split = int(len(shuffled) * 0.8)
    train_idx, test_idx = shuffled[:split], shuffled[split:]

train_keys = set(key.iloc[train_idx].tolist())
test_keys = set(key.iloc[test_idx].tolist())
intersection = train_keys & test_keys

print('Train size:', len(train_idx), '| Test size:', len(test_idx))
print('Nb cles exactes presentes dans train ET test:', len(intersection))
print('Leakage rate estimate (% sur cles test):', round(len(intersection) / max(1, len(test_keys)) * 100, 3))
print('Split backend:', 'scikit-learn' if use_sklearn else 'numpy fallback')


In [ ]:
# Export d'un resume KPI EDA
kpis = {
    'n_rows': int(len(df_rag)),
    'n_cols': int(df_rag.shape[1]),
    'exact_duplicates_pct': round(exact_dup_count / len(df_rag) * 100, 3),
    'near_duplicates_pct': round(near_dup_count / len(df_rag) * 100, 3),
    'doc_tokens_est_mean': float(df_rag['doc_tokens_est'].mean()),
    'doc_tokens_est_p95': float(df_rag['doc_tokens_est'].quantile(0.95)),
    'n_chunks_est_mean': float(df_rag['n_chunks_est'].mean()) if 'n_chunks_est' in df_rag.columns else None,
    'n_chunks_est_p95': float(df_rag['n_chunks_est'].quantile(0.95)) if 'n_chunks_est' in df_rag.columns else None,
}

kpi_path = OUTPUT_DIR / 'eda_kpis.json'
kpi_path.write_text(json.dumps(kpis, indent=2), encoding='utf-8')
print('KPI sauvegardes dans:', kpi_path)
print(json.dumps(kpis, indent=2))


## 5) Actions recommandees avant pipeline RAG

Apres execution du notebook, applique ces decisions selon les resultats:

1. Deduplication:
- supprimer doublons exacts sur `subject+body+answer`
- gerer doublons proches avec une similarite texte (ou hash normalise)

2. Nettoyage:
- normaliser sauts de ligne et espaces
- harmoniser labels de `type`, `queue`, `priority`, `language`

3. Strategie de chunking:
- ajuster `CHUNK_SIZE_CHARS` et `CHUNK_OVERLAP_CHARS` selon p95 de longueur
- valider la granularite par langue

4. Metadata indexing:
- garder `language`, `type`, `queue`, `priority`, `version` comme filtres en base vectorielle
- conserver `tag_*` comme labels de reranking ou filtres secondaires

5. Evaluation retrieval:
- creer un jeu de requetes de validation multi-langue
- mesurer Recall@k / MRR / nDCG apres indexation

Le dossier `../reports/eda_rag/` contient les exports utiles pour industrialiser la suite.
